# Chem 427, Project #11: Nanoparticle size by three techniques

**Introduction:**
In this project we aim to determine if AFM height reads silica nanoparticles smaller than SEM and SAXS do, while reading gold nanoparticles correctly, by more than the combined uncertainty. Additionally, we aim to determine if our own particle segmentation can reproduce the expert-annotated size distribution.

## Data availability

All data are public and were downloaded on 2026-11-08; the MD5 checksum of each zip matches the one listed on its Zenodo record. All cleaning happens in code below, and the notebook does not download anything.

| Files in `data/` | Content | Source | License |
|---|---|---|---|
| `VSL.zip` | VSL laboratory images as 2D arrays | pollen-metrology.com, *nPSize VSL images as 2D arrays*, Zenodo (2021), https://doi.org/10.5281/zenodo.5578927 | CC0 |
| `LNE1.zip`, `LNE2.zip`, `LNE3.zip` | LNE laboratory images as 2D arrays | pollen-metrology.com, *nPSize LNE images as 2D arrays*, Zenodo (2021), https://doi.org/10.5281/zenodo.5578878 | CC0 |
| `annotations.zip` | Expert particle annotations (`annotations.json`) | pollen-metrology.com, *nPSize JSON file: image annotations*, Zenodo (2021), https://doi.org/10.5281/zenodo.5577401 | CC0 |
| `nPSize*.dat` | SAXS curves | Deumer, J., Gollwitzer, C., Zenodo (2022), https://doi.org/10.5281/zenodo.5886834 | CC BY 4.0 |
| `17NRM04_D3_ValidationReport_v1.0_Accepted.pdf` | Validation report with reference sizes | Bartczak, D., Hodoroaba, V.-D., Zenodo (2022), https://doi.org/10.5281/zenodo.7016466 | CC BY 4.0 |

Accessed 2026-11-08.

## Parameters

Constants that encode a scientific or statistical choice, each with its unit and reason. `ASPECT_RATIO_FLAG` is set from the particle shape distributions shown in the section "Particle shape and outlier check". The gold population cutoff will be added here after the histograms in the Exploration section. Running this cell prints nothing.

In [ ]:
import numpy as np

RNG_SEED = 427              # dimensionless; fixed so bootstrap CIs are identical on every rerun
N_BOOTSTRAP = 10_000        # resamples; enough that the CI endpoints change by < ~0.1 nm between seeds
CI_LEVEL = 0.95             # fraction; conventional two-sided confidence level
K_COVERAGE = 2              # dimensionless; coverage factor for expanded uncertainty (~95 %, GUM)
ASPECT_RATIO_FLAG = 1.12    # dimensionless (max Feret / min Feret); above this an EM particle is flagged as probably
                            # elongated or fused. 1.12 is the 99th percentile in silica TEM 2020 (finest pixels, no tail)
                            # and matches the gap in SEM LNE at ~5.6 nm (≈ 1.12 for a 48 nm particle). Flags only; nothing is removed.

rng = np.random.default_rng(RNG_SEED)

## Loading the expert annotations

`annotations.json` (inside `data/annotations.zip`) nests laboratory → microscope → image → particle. Each row of `annotations` below is **one expert-annotated particle in one image**. The columns give its laboratory, technique, material (`npsize`), image name, pixel size (nm/px), AFM height (nm, filled only for AFM), maximum and minimum Feret diameter (nm), and the particle's centroid position in the image (nm, used to check for duplicate particles). Each measurement is stored as a `{unit, value}` pair, so the loader checks that the unit is nm and keeps the value. The cell prints the total row count (expect 11,227) and the count for each lab and technique.

In [ ]:
import json
import zipfile

import pandas as pd

SIZE_FIELDS = {                              # JSON key -> column name (all in nm)
    "afm_height": "afm_height_nm",
    "maximum_feret_diameter": "feret_max_nm",
    "minimum_feret_diameter": "feret_min_nm",
}

with zipfile.ZipFile("data/annotations.zip") as z:
    raw = json.load(z.open("annotations.json"))

rows = []
for lab in raw.values():
    for scope_key, scope in lab.items():
        if not scope_key.startswith("microscope"):    # skip the "name" entry
            continue
        for image in scope["data"].values():
            for particle_key, particle in image["particles"].items():
                if not particle_key.startswith("particle"):   # skip the per-image "total"
                    continue
                row = {
                    "laboratory": lab["name"],
                    "technique": scope["type"],
                    "npsize": image["npsize"],
                    "image": image["name"],
                    "pixel_size_nm": image["pixel_size"],     # nm per pixel
                }
                for key, column in SIZE_FIELDS.items():
                    entry = particle["measurements"][key]
                    if entry["unit"] != "nm":
                        raise ValueError(f"{key} is in {entry['unit']}, not nm")
                    row[column] = entry["value"]
                centroid = particle["measurements"]["centroid"]   # particle position in the image
                if centroid["unit"] != "nm":
                    raise ValueError(f"centroid is in {centroid['unit']}, not nm")
                row["centroid_x_nm"] = centroid["x"]
                row["centroid_y_nm"] = centroid["y"]
                rows.append(row)

annotations = pd.DataFrame(rows)
for column in SIZE_FIELDS.values():
    annotations[column] = pd.to_numeric(annotations[column])   # None -> NaN

print(f"{len(annotations)} annotated particles")
print(annotations.groupby(["laboratory", "technique"]).size())

## Problems found on loading

This cell checks the two materials in scope, nPSize01 (gold) and nPSize10 (silica), for problems that could bias a size comparison, and prints one line per check. It removes nothing. The scientific choice is what counts as a problem: (1) missing size values, (2) pixel size varying between images, since it sets the size resolution, (3) physically impossible values (size ≤ 0 nm, or minimum Feret > maximum Feret), (4) groups with few particles or images, (5) duplicate rows, and (6) format quirks: entries in the JSON that are not microscopes or particles, which the loader skips. A duplicate counts as the same particle only if it has the same image and centroid; identical sizes alone are expected, because sizes are measured in whole pixels. Outliers are checked in "Checks on the result". The full tables per material × technique are in Appendix A1.

Expect six lines: AFM height missing only on non-AFM rows, a fixed AFM pixel size but several per electron-microscopy group, no impossible values, silica AFM as the smallest group, a nonzero count of identical-size rows but zero same-particle duplicates, and the skipped JSON entries.

In [ ]:
MATERIALS = ["01", "10"]                       # nPSize01 = gold, nPSize10 = silica
size_columns = list(SIZE_FIELDS.values())      # all in nm

in_scope = annotations[annotations["npsize"].isin(MATERIALS)]
groups = in_scope.groupby(["npsize", "technique"])
is_afm = in_scope["technique"] == "AFM"

afm_missing_non_afm = in_scope.loc[~is_afm, "afm_height_nm"].isna().sum()
afm_missing_afm = in_scope.loc[is_afm, "afm_height_nm"].isna().sum()
feret_missing = in_scope[["feret_max_nm", "feret_min_nm"]].isna().sum().sum()
print(f"(1) Missing: afm_height_nm on {afm_missing_non_afm} of {(~is_afm).sum()} non-AFM rows (structural) "
      f"and {afm_missing_afm} AFM rows; Feret diameters on {feret_missing} rows")

pixel_sizes = groups["pixel_size_nm"].agg(["min", "max", "nunique"])            # nm/px
em_pixels = pixel_sizes.drop(index="AFM", level="technique")
afm_pixel = in_scope.loc[is_afm, "pixel_size_nm"].unique()
print(f"(2) Pixel size: AFM fixed at {', '.join(f'{p:.2f}' for p in afm_pixel)} nm/px; electron microscopy "
      f"{em_pixels['nunique'].min()}–{em_pixels['nunique'].max()} values per group, "
      f"{em_pixels['min'].min():.2f} to {em_pixels['max'].max():.2f} nm/px")

nonpositive = int((in_scope[size_columns] <= 0).sum().sum())
min_above_max = int((in_scope["feret_min_nm"] > in_scope["feret_max_nm"]).sum())
print(f"(3) Impossible values: {nonpositive} sizes <= 0 nm, {min_above_max} rows with min Feret > max Feret")

group_counts = groups.agg(n_particles=("image", "size"), n_images=("image", "nunique"))
smallest = group_counts["n_particles"].idxmin()
print(f"(4) Smallest group: nPSize{smallest[0]} {smallest[1]}, {group_counts.loc[smallest, 'n_particles']} particles "
      f"in {group_counts.loc[smallest, 'n_images']} images (all groups in A1)")

identity_columns = ["laboratory", "technique", "npsize", "image", "pixel_size_nm"]
same_sizes = in_scope.duplicated(subset=identity_columns + size_columns).sum()
same_particle = in_scope.duplicated(subset=["image", "centroid_x_nm", "centroid_y_nm"]).sum()
print(f"(5) Duplicates: {same_sizes} rows with the same image and identical sizes; "
      f"{same_particle} with the same image and centroid (same particle twice)")

non_microscope_keys = sum(not key.startswith("microscope") for lab in raw.values() for key in lab)
images = [image for lab in raw.values() for key, scope in lab.items()
          if key.startswith("microscope") for image in scope["data"].values()]
non_particle_keys = sum(not key.startswith("particle") for image in images for key in image["particles"])
print(f"(6) Format quirks skipped (all files): {non_microscope_keys} non-microscope entries in a laboratory, "
      f"{non_particle_keys} non-particle entries in {len(images)} images")

## Exploration: descriptive statistics

This cell summarises the size distribution of the three groups used in the comparison, for each material. The scientific choices come from the decisions log:
- **Size measures:** the primary size is the expert AFM height for AFM and the **minimum Feret diameter** for SEM and TEM (D2).
- **Groups:** SEM is pooled over BAM and LNE, the primary reference; TEM is the secondary reference (D1).
- **What is not done yet:** gold (nPSize01) is pooled over its 30 nm and 60 nm populations. The split is decided after the histograms.

Median and IQR are shown because the analysis is median-based. The full table, with SEM per laboratory, TSEM, mean and SD, and the maximum Feret diameter, is in Appendix A2. Expect a 6-row table in nm, with silica AFM at about 44 nm (n = 42) and SEM at about 48 nm, and wide IQRs for gold.

In [ ]:
MATERIAL_NAMES = {"01": "gold", "10": "silica"}


def summarize(sizes):
    """Descriptive statistics of one size distribution (nm)."""
    q1, median, q3 = sizes.quantile([0.25, 0.50, 0.75])
    return {
        "median": median, "Q1": q1, "Q3": q3, "IQR": q3 - q1,
        "mean": sizes.mean(), "SD": sizes.std(),
    }


summary_rows = []
for material in MATERIALS:
    rows_m = in_scope[in_scope["npsize"] == material]
    sem = rows_m[rows_m["technique"] == "SEM"]
    groups_m = {                                  # group label -> (rows, primary size column)
        "AFM": (rows_m[rows_m["technique"] == "AFM"], "afm_height_nm"),
        "SEM (BAM + LNE)": (sem, "feret_min_nm"),
        "  SEM BAM": (sem[sem["laboratory"] == "BAM"], "feret_min_nm"),
        "  SEM LNE": (sem[sem["laboratory"] == "LNE"], "feret_min_nm"),
        "TEM": (rows_m[rows_m["technique"] == "TEM"], "feret_min_nm"),
        "TSEM": (rows_m[rows_m["technique"] == "TSEM"], "feret_min_nm"),
    }
    for label, (rows_g, primary) in groups_m.items():
        stats = summarize(rows_g[primary])
        is_em = primary == "feret_min_nm"         # AFM lateral sizes are tip-dilated
        max_feret = rows_g["feret_max_nm"]
        summary_rows.append({
            "material": f"{MATERIAL_NAMES[material]} ({material})",
            "group": label,
            "size measure": "min Feret" if is_em else "AFM height",
            "n particles": len(rows_g),
            "n images": rows_g["image"].nunique(),
            **stats,
            "max Feret median": max_feret.median() if is_em else float("nan"),
            "max Feret IQR": (max_feret.quantile(0.75) - max_feret.quantile(0.25)) if is_em else float("nan"),
        })

size_summary = pd.DataFrame(summary_rows).set_index(["material", "group"])   # full table: Appendix A2

MAIN_GROUPS = ["AFM", "SEM (BAM + LNE)", "TEM"]   # primary and secondary references (D1)
main_columns = ["n particles", "n images", "median", "IQR"]
is_main = size_summary.index.get_level_values("group").isin(MAIN_GROUPS)
size_summary.loc[is_main, main_columns].round(1)  # sizes in nm

For silica, the median AFM height is 43.7 nm (IQR 3.0 nm, n = 42 in 10 images), compared with a median SEM minimum Feret of 47.7 nm (IQR 2.7 nm, n = 287) and 49.6 nm by TEM (n = 353). For gold, the pooled medians are not comparable between groups: BAM's SEM annotations (per-lab rows in Appendix A2) sit almost entirely in the 60 nm population (median 64.3 nm, IQR 6.2 nm), while LNE, TEM and AFM span both populations (IQR 29 to 30 nm).

## Exploration: size distributions

This figure compares the shape of the size distributions of AFM height, SEM minimum Feret (BAM + LNE, the primary reference) and TEM minimum Feret (the secondary reference), with one panel per material. TSEM and the per-lab split are left out to keep the figure readable; they are in the table above. The scientific choices:
- **Probability density, not counts:** the groups differ in size (n = 42 to 620), so each curve is normalised to unit area.
- **Frequency polygons:** each curve joins the density at each bin centre, so that three overlapping distributions stay readable. Each series has its own line style and marker.
- **Bin width:** 4 nm for gold and 2 nm for silica. Electron-microscopy sizes come in whole-pixel steps of 0.9 to 1.9 nm, and bins narrower than about one step would make a comb-shaped artifact.

Expect two modes near 30 nm and 60 nm for gold, and a single narrow peak near 45 to 50 nm for silica.

In [ ]:
import matplotlib.pyplot as plt

HIST_BIN_WIDTH_NM = {"01": 4.0, "10": 2.0}         # nm; >= ~1 SEM pixel step so whole-pixel Feret values do not make a comb
HIST_RANGE_NM = {"01": (0, 110), "10": (30, 70)}   # nm; covers every in-scope value of the plotted measures

series_style = {                              # label -> (technique, size column, colour, line style, marker)
    "AFM height": ("AFM", "afm_height_nm", "#2a78d6", "-", "o"),
    "SEM min Feret (BAM + LNE)": ("SEM", "feret_min_nm", "#eb6834", "--", "s"),
    "TEM min Feret": ("TEM", "feret_min_nm", "#1baf7a", ":", "^"),
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, material in zip(axes, MATERIALS):
    rows_m = in_scope[in_scope["npsize"] == material]
    low, high = HIST_RANGE_NM[material]
    bin_edges = np.arange(low, high + HIST_BIN_WIDTH_NM[material], HIST_BIN_WIDTH_NM[material])   # nm
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2                                            # nm
    for label, (technique, column, colour, linestyle, marker) in series_style.items():
        sizes = rows_m.loc[rows_m["technique"] == technique, column]
        density, _ = np.histogram(sizes, bins=bin_edges, density=True)   # 1/nm
        ax.plot(bin_centers, density, color=colour, linestyle=linestyle, marker=marker,
                markersize=5, linewidth=1.5, label=f"{label} (n = {len(sizes)})")
    ax.set_title(f"{MATERIAL_NAMES[material]} (nPSize{material})")
    ax.set_xlabel("Particle size (nm)")
    ax.set_ylabel("Probability density (1/nm)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

For gold, all three techniques show two well-separated modes near 30 nm and 60 nm, with few particles between 38 and 50 nm (25 of 620 by AFM, 5 of 318 by SEM, 8 of 422 by TEM). For silica, the AFM height peak (median 43.7 nm) lies about 4 nm below the SEM and TEM peaks, and TEM has a tail from 55 to 65 nm that holds 69 of its 353 particles.

## Checks on the result

These checks ask whether a choice made above could change the AFM − SEM comparison. The supporting tables are in the Appendix.

### Outliers: particle shape flag

A particle seen by electron microscopy is flagged if its aspect ratio (maximum Feret / minimum Feret, dimensionless) exceeds `ASPECT_RATIO_FLAG`, as a probable elongated or fused particle (D4). The scientific choice is a physical flag on shape instead of a statistical fence on size; the shape evidence behind the cutoff is in Appendix A3. AFM particles are not flagged, because tip dilation distorts lateral shape, so the AFM row is blank in the flag columns. No particle is removed, because the analysis is median-based. The table gives each silica group's median of the primary size (nm), the number flagged, the median without the flagged particles, and the change. Gold is checked after its 30 nm and 60 nm populations are split (task 005). Expect a 3-row table in which the flagged particles are a sizeable fraction of each electron-microscopy group.

In [ ]:
silica = in_scope[in_scope["npsize"] == "10"]
silica_em = silica[silica["technique"].isin(["SEM", "TEM"])].copy()
silica_em["aspect_ratio"] = silica_em["feret_max_nm"] / silica_em["feret_min_nm"]   # dimensionless

flag_groups = {                                   # label -> (rows, primary size column)
    "AFM": (silica[silica["technique"] == "AFM"], "afm_height_nm"),
    "SEM (BAM + LNE)": (silica_em[silica_em["technique"] == "SEM"], "feret_min_nm"),
    "TEM (all sessions)": (silica_em[silica_em["technique"] == "TEM"], "feret_min_nm"),
}
flag_rows = []
for label, (rows_g, column) in flag_groups.items():
    row = {"group": label, "n particles": len(rows_g), "median (nm)": rows_g[column].median()}
    if "aspect_ratio" in rows_g:
        flagged = rows_g["aspect_ratio"] > ASPECT_RATIO_FLAG
        row["n flagged"] = int(flagged.sum())
        row["median without flagged (nm)"] = rows_g.loc[~flagged, column].median()
        row["change (nm)"] = row["median without flagged (nm)"] - row["median (nm)"]
    else:                                         # AFM: lateral shape is tip-dilated, so no shape flag
        row.update({"n flagged": np.nan, "median without flagged (nm)": np.nan, "change (nm)": np.nan})
    row["n removed"] = 0
    flag_rows.append(row)
flag_table = pd.DataFrame(flag_rows).set_index("group")
flag_table["n flagged"] = flag_table["n flagged"].astype("Int64")   # whole counts; blank for AFM
flag_table.round(2)                               # nm, except counts

## Appendix

These tables support the checks above and are not needed to follow the argument. Each one is referred to from the main text.

### A1. Loading checks in full

The tables behind checks (1), (2) and (4) in "Problems found on loading", per material × technique. Expect NaN counts per size column, the pixel-size range (nm/px) with the number of distinct values, and the particle and image counts.

In [ ]:
print("(1) Missing values per column (NaN count):")
print(groups[size_columns].agg(lambda s: s.isna().sum()), "\n")
print("(2) Pixel size per group (nm/px):")
print(pixel_sizes, "\n")
print("(4) Particles and images per group:")
print(group_counts)

### A2. Descriptive statistics in full

The full version of the table in "Exploration: descriptive statistics". SEM is shown pooled (BAM + LNE) and per laboratory, so the difference between labs is visible; TSEM is context only (D1). Mean and SD are shown next to the median and IQR to indicate skew. Maximum Feret is shown as a second block for the electron-microscopy rows only (the sensitivity measure in D2); for AFM it is left blank, because tip dilation roughly doubles AFM lateral sizes. Expect a 12-row table in nm.

In [ ]:
size_summary.round(1)                             # all sizes in nm

### A3. Silica by SEM laboratory and TEM session

The CEA TEM images fall into three sessions, read from the year at the start of each image name (2019, 2020, 2021). For each SEM laboratory and TEM session the table gives the pixel size, the number of images and particles, the median and IQR of the minimum Feret diameter (nm), and percentiles of the aspect ratio (dimensionless). It shows whether one session or lab differs as a whole, which a per-particle rule cannot detect, and the shape evidence behind `ASPECT_RATIO_FLAG`: the 99th percentile in TEM 2020, the session with the finest pixels and no large-size tail (D4). Expect TEM 2019 to stand apart from 2020 and 2021 in both size and shape.

In [ ]:
silica_em["subset"] = "SEM " + silica_em["laboratory"]
is_tem = silica_em["technique"] == "TEM"
silica_em.loc[is_tem, "subset"] = "TEM " + silica_em.loc[is_tem, "image"].str[:4]   # session = year in the image name

subset_rows = []
for subset, rows_s in silica_em.groupby("subset"):
    q1, median, q3 = rows_s["feret_min_nm"].quantile([0.25, 0.50, 0.75])
    aspect = rows_s["aspect_ratio"].quantile([0.50, 0.90, 0.95, 0.99])
    subset_rows.append({
        "subset": subset,
        "pixel size (nm/px)": ", ".join(f"{p:.2f}" for p in sorted(rows_s["pixel_size_nm"].unique())),
        "n images": rows_s["image"].nunique(),
        "n particles": len(rows_s),
        "min Feret median (nm)": median,
        "min Feret IQR (nm)": q3 - q1,
        "aspect ratio p50": aspect[0.50],
        "p90": aspect[0.90],
        "p95": aspect[0.95],
        "p99": aspect[0.99],
    })
pd.DataFrame(subset_rows).set_index("subset").round(2)   # nm, nm/px, aspect ratio dimensionless

### A4. Bootstrap unit: particles or images?

This cell compares two bootstrap 95% CIs for the silica (nPSize10) medians and the AFM − SEM median difference, using the primary measures (AFM height, SEM minimum Feret, BAM + LNE). The scientific choice is the **resampling unit**:
- **Particle bootstrap:** each resample draws particles with replacement. This assumes every particle is independent.
- **Image (cluster) bootstrap:** each resample draws whole images with replacement and keeps all their particles. This allows particles in one image to share errors (tip, z-calibration, background fit for AFM; focus and threshold for SEM).

In both versions SEM is resampled within each laboratory (BAM, LNE), so each resample keeps the same share of each lab and only the resampling unit differs. AFM and SEM are resampled independently, and the difference is taken resample by resample. Gold is not included until its 30 nm and 60 nm populations are split. Expect a table in nm with the median, each CI and its width, and the ratio of image-CI width to particle-CI width. A ratio near 1 means images do not differ much; a ratio clearly above 1 means the particle CI is too narrow.

In [ ]:
def resampling_units(rows, column, unit):
    """Split one group's sizes (nm) into strata (laboratories), each a list of resampling units.

    unit = "particle": every particle is its own unit; unit = "image": all particles of one image form a unit.
    """
    strata = []
    for _, rows_lab in rows.groupby("laboratory"):
        if unit == "image":
            strata.append([rows_img[column].to_numpy() for _, rows_img in rows_lab.groupby("image")])
        else:
            strata.append([np.array([size]) for size in rows_lab[column]])
    return strata


def bootstrap_medians(strata, n_boot, rng):
    """Median (nm) of each of n_boot resamples; units are drawn with replacement within each stratum."""
    medians = np.empty(n_boot)
    for b in range(n_boot):
        picked = [units[i] for units in strata for i in rng.integers(0, len(units), len(units))]
        medians[b] = np.median(np.concatenate(picked))
    return medians


silica = in_scope[in_scope["npsize"] == "10"]
afm_silica = silica[silica["technique"] == "AFM"]
sem_silica = silica[silica["technique"] == "SEM"]           # BAM + LNE, primary reference (D1)
estimates = {                                               # nm
    "AFM median": afm_silica["afm_height_nm"].median(),
    "SEM median": sem_silica["feret_min_nm"].median(),
}
estimates["AFM − SEM"] = estimates["AFM median"] - estimates["SEM median"]

tail = (1 - CI_LEVEL) / 2
ci = {}                                                     # unit -> quantity -> (low, high) in nm
for unit in ("particle", "image"):
    afm_boot = bootstrap_medians(resampling_units(afm_silica, "afm_height_nm", unit), N_BOOTSTRAP, rng)
    sem_boot = bootstrap_medians(resampling_units(sem_silica, "feret_min_nm", unit), N_BOOTSTRAP, rng)
    boots = {"AFM median": afm_boot, "SEM median": sem_boot, "AFM − SEM": afm_boot - sem_boot}
    ci[unit] = {name: np.quantile(values, [tail, 1 - tail]) for name, values in boots.items()}

ci_rows = []
for name, estimate in estimates.items():
    row = {"quantity": name, "estimate (nm)": estimate}
    for unit in ("particle", "image"):
        low, high = ci[unit][name]
        row[f"{unit} CI low"] = low
        row[f"{unit} CI high"] = high
        row[f"{unit} CI width"] = high - low
    row["width ratio (image / particle)"] = row["image CI width"] / row["particle CI width"]
    ci_rows.append(row)

bootstrap_unit_comparison = pd.DataFrame(ci_rows).set_index("quantity")
bootstrap_unit_comparison.round(2)                          # nm, except the dimensionless ratio

## Environment

This cell prints the Python version and the versions of the packages used above, so the results can be reproduced. Expect Python 3.14 and the versions locked in `uv.lock`.

In [ ]:
import sys
import matplotlib

print(f"Python {sys.version.split()[0]}")
for module in (np, pd, matplotlib):
    print(f"{module.__name__} {module.__version__}")